# Phase 1: Train Shared NAR on CLRS-30 Graph Algorithms

Train a single MPNN-based NAR model on multiple graph algorithms simultaneously.
The processor is shared across all algorithms; the encoder/decoder adapt dynamically
based on each algorithm's output/hint types.

**Algorithms:** BFS, DFS, Dijkstra, Bellman-Ford, MST Prim, MST Kruskal,
Articulation Points, Bridges, Fast MIS, Eccentricity

In [ ]:
import sys
from pathlib import Path

# Handle import paths
if Path("../data").exists() and ".." not in sys.path:
    sys.path.insert(0, "..")
elif Path("data").exists() and "." not in sys.path:
    sys.path.insert(0, ".")

try:
    import torch_geometric.data.data as _pyg_data
    if not hasattr(_pyg_data, 'DataEdgeAttr'):
        _pyg_data.DataEdgeAttr = type('DataEdgeAttr', (), {})
        _pyg_data.DataTensorAttr = type('DataTensorAttr', (), {})
except Exception:
    pass

import os
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt
import numpy as np

from data import (
    AVAILABLE_ALGORITHMS,
    get_clrs_dataloader,
    get_algorithm_spec,
    spec_to_model_types,
    batch_to_model_inputs,
    MultiAlgorithmLoader,
)
from models import NARModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

SEED = 42
torch.manual_seed(SEED)

## Configuration

In [ ]:
# --- Algorithm selection ---
# All 10 available CLRS-30 graph algorithms
ALGORITHMS = AVAILABLE_ALGORITHMS
print(f"Training on {len(ALGORITHMS)} algorithms: {ALGORITHMS}")

# === Scale toggle ===
LOCAL_DEBUG = False

if LOCAL_DEBUG:
    HIDDEN_DIM = 32
    NUM_LAYERS = 2
    NUM_HEADS = 4
    NAR_EPOCHS = 10
    TRAIN_SAMPLES = 100
    VAL_SAMPLES = 32
else:
    HIDDEN_DIM = 128
    NUM_LAYERS = 4
    NUM_HEADS = 8
    NAR_EPOCHS = 100
    TRAIN_SAMPLES = 1000
    VAL_SAMPLES = 128

PROCESSOR_TYPE = "mpnn"
NAR_BATCH_SIZE = 32
NAR_LR = 1e-3
NUM_NODES = 16
EDGE_PROB = 0.2

# Paths
IN_COLAB = "COLAB_GPU" in os.environ
REPO_ROOT = Path("..") if Path("../data").exists() else Path(".")
if IN_COLAB:
    DRIVE_ROOT = Path("/content/drive/MyDrive/nar-mechinterp")
    CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / "multi_algo"
else:
    CHECKPOINT_DIR = REPO_ROOT / "checkpoints" / "multi_algo"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Mode: {'LOCAL DEBUG' if LOCAL_DEBUG else 'FULL SCALE'}")
print(f"NAR: hidden_dim={HIDDEN_DIM}, layers={NUM_LAYERS}, heads={NUM_HEADS}")
print(f"Training: {NAR_EPOCHS} epochs, {TRAIN_SAMPLES} samples/algo")
print(f"Checkpoints: {CHECKPOINT_DIR}")

## Load Datasets

Create a `MultiAlgorithmLoader` for train and val splits. Each yields
`(batch, algo_name, output_types, hint_types)` in round-robin order.

In [ ]:
train_loader = MultiAlgorithmLoader(
    algorithms=ALGORITHMS,
    split="train",
    batch_size=NAR_BATCH_SIZE,
    num_samples=TRAIN_SAMPLES,
    num_nodes=NUM_NODES,
    edge_probability=EDGE_PROB,
    seed=SEED,
    shuffle=True,
)

val_loader = MultiAlgorithmLoader(
    algorithms=ALGORITHMS,
    split="val",
    batch_size=NAR_BATCH_SIZE,
    num_samples=VAL_SAMPLES,
    num_nodes=NUM_NODES,
    edge_probability=EDGE_PROB,
    seed=SEED + 1,
    shuffle=False,
)

print(f"Train: {len(train_loader)} total batches across {train_loader.num_algorithms} algorithms")
print(f"Val:   {len(val_loader)} total batches")
print(f"\nPer-algorithm specs:")
for algo in ALGORITHMS:
    ot, ht = train_loader.output_types[algo], train_loader.hint_types[algo]
    print(f"  {algo:25s} outputs={list(ot.keys()):30s} hints={list(ht.keys())}")

## Create NAR Model

In [ ]:
model = NARModel(
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    processor_type=PROCESSOR_TYPE,
).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"NAR model parameters: {num_params:,}")

## Training Functions

In [ ]:
def train_epoch(model, loader, optimizer, device):
    """Train one epoch over all algorithms (round-robin)."""
    model.train()
    total_loss = 0.0
    algo_losses = {algo: [] for algo in ALGORITHMS}
    n_batches = 0

    for batch, algo, output_types, hint_types in loader:
        batch = batch.to(device)
        spec = loader.specs[algo]
        inputs, outputs, hints = batch_to_model_inputs(batch, spec, device)

        optimizer.zero_grad()
        output = model(
            inputs=inputs, hints=hints, outputs=outputs,
            output_types=output_types, hint_types=hint_types,
            num_steps=batch.lengths.max().item(),
        )
        output.total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        loss_val = output.total_loss.item()
        total_loss += loss_val
        algo_losses[algo].append(loss_val)
        n_batches += 1

    avg_loss = total_loss / max(n_batches, 1)
    per_algo = {a: np.mean(v) if v else float('nan') for a, v in algo_losses.items()}
    return avg_loss, per_algo


@torch.no_grad()
def validate(model, loader, device):
    """Validate over all algorithms, return per-algorithm accuracy."""
    model.eval()
    total_loss = 0.0
    algo_accs = {algo: [] for algo in ALGORITHMS}
    n_batches = 0

    for batch, algo, output_types, hint_types in loader:
        batch = batch.to(device)
        spec = loader.specs[algo]
        inputs, outputs, hints = batch_to_model_inputs(batch, spec, device)

        output = model(
            inputs=inputs, outputs=outputs,
            output_types=output_types, hint_types=hint_types,
            num_steps=batch.lengths.max().item(),
        )
        total_loss += output.total_loss.item()
        n_batches += 1

        # Per-algorithm accuracy
        for name, pred in output.predictions.items():
            if name not in outputs:
                continue
            target = outputs[name]
            otype = output_types.get(name, '')
            if otype == 'node_mask':
                pred_bin = torch.sigmoid(pred[:, :target.shape[-1]]) > 0.5
                acc = (pred_bin == target.bool()).float().mean()
            elif otype == 'node_pointer':
                pred_idx = pred.argmax(-1)
                tgt_idx = target.long() if target.dim() < pred.dim() else target.argmax(-1)
                acc = (pred_idx == tgt_idx).float().mean()
            elif otype == 'edge_mask':
                if pred.shape == target.shape:
                    pred_bin = torch.sigmoid(pred) > 0.5
                    acc = (pred_bin == target.bool()).float().mean()
                else:
                    acc = torch.tensor(0.0)
            else:
                acc = torch.tensor(0.0)
            algo_accs[algo].append(acc.item())

    avg_loss = total_loss / max(n_batches, 1)
    per_algo = {a: np.mean(v) if v else float('nan') for a, v in algo_accs.items()}
    avg_acc = np.nanmean(list(per_algo.values()))
    return avg_loss, per_algo, avg_acc

## Training Loop

In [ ]:
_ckpt_path = CHECKPOINT_DIR / "best.pt"

if _ckpt_path.exists():
    ckpt = torch.load(_ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded existing checkpoint (epoch {ckpt['epoch']}, avg_acc={ckpt.get('avg_acc', 'N/A')})")
    print("Skipping training. Delete checkpoint to retrain.")
    _skip_training = True
else:
    _skip_training = False

if not _skip_training:
    optimizer = optim.AdamW(model.parameters(), lr=NAR_LR, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=NAR_EPOCHS)

    history = {
        'train_losses': [], 'val_losses': [], 'avg_accs': [],
        'per_algo_loss': [], 'per_algo_acc': [],
    }
    best_loss = float('inf')

    for epoch in range(NAR_EPOCHS):
        train_loss, algo_train_loss = train_epoch(model, train_loader, optimizer, device)
        val_loss, algo_val_acc, avg_acc = validate(model, val_loader, device)
        scheduler.step()

        history['train_losses'].append(train_loss)
        history['val_losses'].append(val_loss)
        history['avg_accs'].append(avg_acc)
        history['per_algo_loss'].append(algo_train_loss)
        history['per_algo_acc'].append(algo_val_acc)

        if val_loss < best_loss:
            best_loss = val_loss
            torch.save({
                'model_state_dict': model.state_dict(),
                'epoch': epoch,
                'val_loss': val_loss,
                'avg_acc': avg_acc,
                'per_algo_acc': algo_val_acc,
                'config': {
                    'hidden_dim': HIDDEN_DIM, 'num_layers': NUM_LAYERS,
                    'num_heads': NUM_HEADS, 'processor_type': PROCESSOR_TYPE,
                    'algorithms': ALGORITHMS,
                },
            }, CHECKPOINT_DIR / "best.pt")

        # Print summary every 10 epochs
        if (epoch + 1) % 10 == 0 or epoch == 0:
            algo_str = "  ".join(f"{a[:6]}={algo_val_acc.get(a, 0):.2f}" for a in ALGORITHMS[:5])
            print(f"Epoch {epoch+1:3d}/{NAR_EPOCHS} | "
                  f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
                  f"Avg Acc: {avg_acc:.3f} | {algo_str} ...")

    # Save final model and history
    torch.save({'model_state_dict': model.state_dict()}, CHECKPOINT_DIR / "final.pt")
    torch.save(history, CHECKPOINT_DIR / "training_history.pt")
    print(f"\nBest val loss: {best_loss:.4f}")
    print(f"Saved to {CHECKPOINT_DIR}")

## Training Curves

In [ ]:
_hist_path = CHECKPOINT_DIR / "training_history.pt"
if _hist_path.exists():
    history = torch.load(_hist_path, weights_only=False)
    print(f"Loaded training history ({len(history['train_losses'])} epochs)")
elif 'history' not in dir():
    print("No training history available. Run training first.")
    history = None

if history is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Loss curves
    axes[0].plot(history['train_losses'], label="Train")
    axes[0].plot(history['val_losses'], label="Val")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title("Training Loss (all algorithms)")
    axes[0].legend(); axes[0].set_yscale("log")

    # Average accuracy
    axes[1].plot(history['avg_accs'])
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
    axes[1].set_title("Average Validation Accuracy")
    axes[1].set_ylim(0, 1.05)

    # Per-algorithm accuracy at final epoch
    if history['per_algo_acc']:
        final_accs = history['per_algo_acc'][-1]
        algos = sorted(final_accs.keys())
        accs = [final_accs[a] for a in algos]
        axes[2].barh(algos, accs, color='steelblue')
        axes[2].set_xlabel("Accuracy")
        axes[2].set_title("Final Per-Algorithm Accuracy")
        axes[2].set_xlim(0, 1.05)
        for i, v in enumerate(accs):
            axes[2].text(v + 0.01, i, f"{v:.2f}", va='center', fontsize=9)

    plt.tight_layout()
    plt.show()

## Per-Algorithm Accuracy Over Time

In [ ]:
if history is not None and history['per_algo_acc']:
    n_algos = len(ALGORITHMS)
    n_cols = min(5, n_algos)
    n_rows = (n_algos + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
    axes = np.array(axes).flatten()

    for i, algo in enumerate(ALGORITHMS):
        acc_over_time = [epoch_accs.get(algo, float('nan'))
                         for epoch_accs in history['per_algo_acc']]
        axes[i].plot(acc_over_time)
        axes[i].set_title(algo, fontsize=10)
        axes[i].set_ylim(0, 1.05)
        axes[i].set_xlabel("Epoch")

    # Hide unused subplots
    for i in range(n_algos, len(axes)):
        axes[i].set_visible(False)

    fig.suptitle("Per-Algorithm Validation Accuracy", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## Summary

The trained NAR checkpoint is saved at the path above.
Use `02_train_sae.ipynb` to collect activations and train SAEs.